In [ ]:
import torch, random, io, sys, warnings
import os, numpy as np, pandas as pd, pyreadr
from tqdm import tqdm

import sys, os
sys.path.append(os.path.abspath(".."))

from cpd_model import parse_args, learn_one_seq_penalty

warnings.filterwarnings("ignore")

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# ======================================================
# CONFIG
# ======================================================
REPS = 10
TRUE_CP = [100, 200]

KAPPA_LIST = [0.2, 1.5]
TOL = 10

# ======================================================
# BASE ARGS
# ======================================================
base_args = parse_args()
base_args.epoch = 150
base_args.K_dim = 2
base_args.z_dim = 3
base_args.decoder_lr = 0.01
base_args.decoder_iteration = 20
base_args.langevin_s = 0.2
base_args.langevin_K = 100
base_args.kappa = 0.8
base_args.penalties = [0.01, 0.05, 0.1, 1]
base_args.nu_iteration = 20
base_args.output_layer = [50, 50]
base_args.scale_delta = False
base_args.signif_level = 0.99
base_args.true_CP_full = TRUE_CP

# ======================================================
# MAIN LOOP
# ======================================================
records = []

GLOBAL_SEED = 1

for kappa in KAPPA_LIST:

    print(f"\n==============================")
    print(f" Running kappa = {kappa}")
    print(f"==============================")

    for rep in range(1, REPS + 1):

        print(f"\n--- Replicate {rep}/{REPS} ---")

        SEED = GLOBAL_SEED + rep
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

        Y = pyreadr.read_r(f"../real_data_sim/sim_dat_ult_5_{rep}.RDS")
        X = pyreadr.read_r(f"../real_data_sim/sim_x_ult_5_{rep}.RDS")

        Y_df = np.array(list(Y.values())[0])
        X_df = np.array(list(X.values())[0])

        X_rep = np.repeat(X_df[:, np.newaxis, :], 100, axis=1)
        Y = Y_df[:, :, 0:3]
        X = X_rep

        args = parse_args()
        args.__dict__.update(base_args.__dict__)

        args.kappa = kappa
        args.x_dim = X.shape[2]
        args.y_dim = Y.shape[2]
        args.num_time = X.shape[0]
        args.num_samples = X.shape[1]

        x_input = torch.tensor(X, dtype=torch.float32).to(device)
        y_input = torch.tensor(Y, dtype=torch.float32).to(device)

        odd_idx = range(1, args.num_time, 2)
        even_idx = range(0, args.num_time, 2)

        x_train = x_input[odd_idx].reshape(-1, args.x_dim)
        x_test  = x_input[even_idx].reshape(-1, args.x_dim)
        y_train = y_input[odd_idx].reshape(-1, args.y_dim)
        y_test  = y_input[even_idx].reshape(-1, args.y_dim)

        results_half = []

        for penalty in args.penalties:
            _stdout = sys.stdout
            # sys.stdout = io.StringIO()
            try:
                loss, pen = learn_one_seq_penalty(
                    args, x_train, y_train, x_test, y_test,
                    penalty=penalty, half=True
                )
            finally:
                sys.stdout = _stdout

            results_half.append([loss, pen])

        results_half = np.array(results_half)
        best_idx = np.argmin(results_half[:, 0])
        best_penalty = args.penalties[best_idx]

        print(f"[INFO] Best penalty = {best_penalty}")

        _stdout = sys.stdout
        # sys.stdout = io.StringIO()
        try:
            out = learn_one_seq_penalty(
                args,
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                penalty=best_penalty,
                half=False
            )
            result = out[0]
        finally:
            sys.stdout = _stdout

        torch.cuda.empty_cache()

        est_cp = np.array(result[5], dtype=int) if len(result[5]) > 0 else np.array([])
        true_cp = np.array(TRUE_CP)

        if len(est_cp) == 0:
            cover_rate = 0
            avg_dist = np.nan
            FP = 0
            FN = len(true_cp)
        else:
            dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])
            min_dist_true = dist_mat.min(axis=0)
            min_dist_est  = dist_mat.min(axis=1)

            cover_rate = np.mean(min_dist_true <= TOL)
            avg_dist   = np.mean(min_dist_true)
            FP = np.sum(min_dist_est > TOL)
            FN = np.sum(min_dist_true > TOL)
        
        records.append({
            "kappa": kappa,
            "rep": rep,
            "best_penalty": best_penalty,
            "num_detected": len(est_cp),

            # core output
            "est_CP": str(list(est_cp)),
            "true_CP": str(TRUE_CP),

            # optional debug
            "CE": result[0],
        })

df = pd.DataFrame(records)
df.to_csv("cpd_kappa_experiment_new.csv", index=False)


In [ ]:
import numpy as np
import pandas as pd
import re
import ast

# =====================================================
# CONFIG
# =====================================================
CSV_PATH = "cpd_kappa_experiment.csv"
TRUE_CP = [100, 200]
TOL = 10
T = 296


# =====================================================
# parse est_CP
# =====================================================
def parse_cp(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if pd.isna(x):
        return []
    try:
        return list(ast.literal_eval(str(x)))
    except:
        return []


# =====================================================
# evaluation
# =====================================================
def evaluate_cp(est_cp, true_cp, tol, T):
    est_cp = np.array(est_cp, dtype=int)
    true_cp = np.array(true_cp, dtype=int)

    if len(est_cp) == 0:
        FP = 0
        FN = len(true_cp)
        Dt_e = np.nan 
        De_t = np.nan
        CE = abs(len(est_cp) - len(true_cp))
        CS = 0.0
        return FP, FN, Dt_e, De_t, CE, CS

    dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])

    min_dist_true = dist_mat.min(axis=0)
    min_dist_est  = dist_mat.min(axis=1)

    FP = np.sum(min_dist_est > tol)
    FN = np.sum(min_dist_true > tol)

    Dt_e = np.max(min_dist_true)
    De_t = np.max(min_dist_est)

    CE = abs(len(est_cp) - len(true_cp))

    # segmentation score
    def get_segments(cp, T):
        cp = np.sort(cp)
        segs = []
        prev = 0
        for c in cp:
            segs.append(set(range(prev, c)))
            prev = c
        segs.append(set(range(prev, T)))
        return segs

    G  = get_segments(true_cp, T)
    Gp = get_segments(est_cp, T)

    def jaccard(A, B):
        return len(A & B) / len(A | B) if len(A | B) > 0 else 0

    CS = 0
    for A in G:
        best = max(jaccard(A, Ap) for Ap in Gp)
        CS += len(A) * best
    CS /= T

    return FP, FN, Dt_e, De_t, CE, CS


# =====================================================
# LOAD
# =====================================================
df = pd.read_csv(CSV_PATH)

df["est_CP"] = df["est_CP"].apply(parse_cp)
df = df.drop(columns=["CE"], errors="ignore")


# =====================================================
# COMPUTE METRICS
# =====================================================
metrics = df["est_CP"].apply(
    lambda cp: evaluate_cp(cp, TRUE_CP, TOL, T)
)

metrics_df = pd.DataFrame(
    metrics.tolist(),
    columns=["FP", "FN", "Dt_e", "De_t", "CE", "CS"]
)

df = pd.concat([df, metrics_df], axis=1)


# =====================================================
# SUMMARY
# =====================================================
summary = (
    df.groupby("kappa")[["FP","FN","Dt_e","De_t","CE","CS"]]
    .mean()
    .round(2)
    .reset_index()
)

print(summary)


# =====================================================
# LATEX TABLE
# =====================================================
def to_latex_table(summary):
    rows = []

    best_FP = summary["FP"].min()
    best_FN = summary["FN"].min()
    best_Dte = summary["Dt_e"].min()
    best_Det = summary["De_t"].min()
    best_CE = summary["CE"].min()
    best_CS = summary["CS"].max()

    for _, r in summary.iterrows():

        def bold(val, best):
            if pd.notna(val) and val == best:
                return f"\\textbf{{{val:.2f}}}"
            return f"{val:.2f}"

        row = [
            r["kappa"],
            bold(r["FP"], best_FP),
            bold(r["FN"], best_FN),
            bold(r["Dt_e"], best_Dte),
            bold(r["De_t"], best_Det),
            bold(r["CE"], best_CE),
            bold(r["CS"], best_CS),
        ]
        rows.append(row)

    latex = []
    latex.append("\\begin{table}[!ht]")
    latex.append("\\caption{Sensitivity analysis with respect to $\\rho$ (kappa).}")
    latex.append("\\label{tab:kappa}")
    latex.append("\\centering")
    latex.append("\\small")
    latex.append("\\begin{tabular}{cccccccc}")
    latex.append("\\toprule")
    latex.append("$\\rho$ & FP$\\downarrow$ & FN$\\downarrow$ & $D_{t\\to e}$ & $D_{e\\to t}$ & CE$\\downarrow$ & CS$\\uparrow$ \\\\")
    latex.append("\\midrule")

    for r in rows:
        latex.append(f"{r[0]:.2f} & {r[1]} & {r[2]} & {r[3]} & {r[4]} & {r[5]} & {r[6]} \\\\")

    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")

    return "\n".join(latex)


latex_table = to_latex_table(summary)

print("\n===== LaTeX Table =====\n")
print(latex_table)

   kappa   FP   FN  Dt_e  De_t   CE    CS
0    0.5  0.8  0.6  34.9  36.6  0.6  0.80
1    0.8  1.0  0.0   0.2  61.9  1.0  0.90
2    2.0  1.0  0.4  32.4  65.4  0.8  0.79

===== LaTeX Table =====

\begin{table}[!ht]
\caption{Sensitivity analysis with respect to $\rho$ (kappa).}
\label{tab:kappa}
\centering
\small
\begin{tabular}{cccccccc}
\toprule
$\rho$ & FP$\downarrow$ & FN$\downarrow$ & $D_{t\to e}$ & $D_{e\to t}$ & CE$\downarrow$ & CS$\uparrow$ \\
\midrule
0.50 & \textbf{0.80} & 0.60 & 34.90 & \textbf{36.60} & \textbf{0.60} & 0.80 \\
0.80 & 1.00 & \textbf{0.00} & \textbf{0.20} & 61.90 & 1.00 & \textbf{0.90} \\
2.00 & 1.00 & 0.40 & 32.40 & 65.40 & 0.80 & 0.79 \\
\bottomrule
\end{tabular}
\end{table}
